# Legislative Proposal Classification – Feature Engineering & Model Evaluation
**Goal:** Improve recall/F1 for the *rejected* class by  
1. Adding Doc2Vec embeddings  
2. Adding distance-to-centroid features from a binary cluster  
3. Retesting baseline models with imbalance treatment  


In [2]:
# Core imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight

# Models
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

# Doc2Vec & Clustering
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.cluster import KMeans

# Imbalance handling
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler


In [5]:
# Change the path to your CSV
df = pd.read_csv("new_nb/Global Vote Prediction Table/votacoes_selected.csv")

# Quick sanity check
display(df.head())
print(df.aprovacao.value_counts())   # label: 1 = approved, 0 = rejected

,id,idDeputadoAutor,legislatura,data,tema,classified_llm,authors_pop,authors_major_comm,orientacao_GOV,aprovacao
0,340896-3,74399,53,2007-02-14,Defesa e Segurança,Approval of Requests,0.0,0,Neutro,1
1,340896-3,74399,53,2007-02-14,Direito Penal e Processual Penal,Approval of Requests,0.0,0,Neutro,1
2,340368-20,74517,53,2007-02-15,"Política, Partidos e Eleições",Change in Processing Regime,0.0,1,Neutro,1
3,340368-21,74399,53,2007-02-15,"Política, Partidos e Eleições",Approval of Requests,0.0,0,Neutro,1
4,340368-31,74324,53,2007-02-15,"Política, Partidos e Eleições",Approval of Final Wording,0.0,0,Neutro,1


aprovacao
1    30280
0    13119
Name: count, dtype: int64


In [ ]:
import re
import nltk
nltk.download('punkt')

def clean_text(text):
    """Basic lowercase + punctuation removal – tweak as needed."""
    return re.sub(r"[^a-zA-ZÀ-ÿ\s]", " ", str(text).lower())

df["clean_text"] = df["proposal_text"].apply(clean_text)

# Prepare TaggedDocument objects for Doc2Vec
tagged_docs = [TaggedDocument(words=doc.split(), tags=[i]) 
               for i, doc in enumerate(df.clean_text)]

# Train Doc2Vec (small config for demo – increase epochs/vector_size later)
d2v_model = Doc2Vec(vector_size=100, min_count=2, 
                    workers=4, epochs=20, dm=1)
d2v_model.build_vocab(tagged_docs)
d2v_model.train(tagged_docs, total_examples=d2v_model.corpus_count, 
                epochs=d2v_model.epochs)

# Store embeddings
df_d2v = pd.DataFrame([d2v_model.infer_vector(doc.words) 
                       for doc in tagged_docs])
df_d2v.columns = [f"d2v_{i}" for i in range(df_d2v.shape[1])]

# Concatenate back to main DataFrame
df = pd.concat([df, df_d2v], axis=1)
